In [1]:
import pandas as pd
import os
import zipfile
import urllib.request
from html.parser import HTMLParser

In [2]:
class HrefParser(HTMLParser):
  def __init__(self):
    super().__init__()
    self.items = []
  def handle_starttag(self, tag, attrs):
    if tag == 'a':
      for name, value in attrs:
        if name == 'href':
          if value.endswith('.zip'):
            self.items.append(value)
  def handle_endtag(self, tag):
    pass
  def handle_data(self, data):
    pass


href_parser = HrefParser()
repo_url = "https://ftp.pride.ebi.ac.uk/pride/data/archive/2017/02/PXD004732/"

In [3]:
try:
    with urllib.request.urlopen(repo_url) as response:
        html_content = response.read().decode('utf-8')
        href_parser.feed(html_content)
except urllib.error.URLError as e:
    print(f"Error accessing URL: {e.reason}")

In [5]:
with open("zip_files_to_download.txt", "w") as file:
    for item in href_parser.items:
        file.write(item + "\n")

In [6]:
def process_single_zip(zip_file_path):
    """
    Extract and remove the zip file.
    """
    FILES_TO_EXTRACT = ["peptides.txt", "msms.txt", "evidence.txt"]
    folder_name = os.path.splitext(zip_file_path)[0]

    # Create the directory if it doesn't exist
    os.makedirs(folder_name, exist_ok=True)

    # Unzip the file into the new directory
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        for file_name in FILES_TO_EXTRACT:
            zip_ref.extract(file_name, folder_name)
    os.remove(zip_file_path)

In [11]:
zip_files = href_parser.items[:504]

In [ ]:
for zip_file in zip_files:
  try:
    urllib.request.urlretrieve(repo_url + zip_file, zip_file)
  except urllib.error.URLError as e:
    print(f"Error accessing URL: {e.reason}")
  process_single_zip(zip_file)

# Create datatset

In [5]:
def make_precursor_column(msms_df):
    precursor_id_column = msms_df.apply(lambda row : row["Modified sequence"]
                  .replace("_", "")
                  .replace("C", "C(UniMod:4)") # fixed carba ptm at Cysteine
                  .replace("ox", "UniMod:35") # M(ox) -> M(UniMod:35)
                  + str(row["Charge"]), 
                  axis=1)
    return precursor_id_column

In [ ]:
FEATS_LIST = ["Sequence", "Length", "m/z", "Charge", "Retention time",
                  "Fragmentation", "PEP", "Score", "Matches", "Intensities", "Masses"]

DATA_DIR = "tsv_folders"
lookup_params = {
    "start id" : 0,
    "ms/ms per precursor" : 20,
    "ref table" : 0,
    "precursors per table": 1000
}

msms_dfs_dicts_list = [{}] # [{"precursor" : dataframe}, {...}]
lookup_dicts = {} # {"precursor" : a dict abt where to find the precursor}
for item in href_parser.items[:80]:
    data_subdir = item.split(".")[0]

    # Only looking for "CID" fragmentation (folder name contains 2xIT_2xHCD, DDA)
    if all([s not in item for s in ["2xIT_2xHCD", "DDA"]]): 
        continue
    
    print(f"Processing {data_subdir}")
    # Read file
    msms_df = pd.read_csv(
        os.path.join(DATA_DIR, data_subdir, "msms.txt"), 
        sep="\t", 
        dtype={"Oxidation (M) site IDs" : object,
               "Oxidation (M) Probabilities" : object,
               "Oxidation (M) Score Diffs" : object,
               "Protein group IDs" : object,
               "Reverse" : object,
               }
        )
    # Filter for CID fragmentation and PEP < 0.01 and not a reverse quence
    msms_df.query("(`Fragmentation` == 'CID') & (`PEP` < 1e-2) & (`Reverse` != '+')", inplace=True) #TODO: matchs.notnull()

    # Create precursor column and make a list of unique precursors in the df
    # Select final features + "Modified sequence" for making "Precursor" column
    msms_df = msms_df[FEATS_LIST + ["Modified sequence"]]
    msms_df["Precursor"] = make_precursor_column(msms_df=msms_df)
    unique_precursor_list = msms_df["Precursor"].drop_duplicates().tolist()

    # Make lookup dict
    for precursor in unique_precursor_list:
        precursor_df = msms_df.query("`Precursor` == @precursor")
        top_n = lookup_params["ms/ms per precursor"]
        if lookup_dicts.get(precursor) == None: # First time this precursor sequence appears
            lookup_dicts[precursor] = {
                "Reference table" : f"{lookup_params["ref table"]:04d}.tsv",
                "Start id" : lookup_params["start id"],
                "Number of MS/MS" : 0, 
                }
            ref_index = lookup_params['ref table']
            msms_dfs_dicts_list[ref_index][precursor] = precursor_df.sort_values(by="PEP").head(top_n)
            # id_list = [lookup_dicts[precursor]['Start id'] + i for i in range (msms_dfs_dicts_list[ref_index][precursor].shape[0])]
            # msms_dfs_dicts_list[ref_index][precursor]['id'] = id_list

            lookup_params['start id'] += top_n
            if lookup_params['start id'] == top_n * lookup_params["precursors per table"]:
                lookup_params['start id'] = 0
                lookup_params['ref table'] += 1
                msms_dfs_dicts_list.append({})

        else: 
            ref_index = int(lookup_dicts[precursor]['Reference table'].split('.')[0])
            dfs_be_merged = [precursor_df, msms_dfs_dicts_list[ref_index][precursor]]
            msms_dfs_dicts_list[ref_index][precursor] = pd.concat(dfs_be_merged, ignore_index=True).sort_values(by="PEP").head(top_n)
            # id_list = [lookup_dicts[precursor]['Start index'] + i for i in range (msms_dfs_dicts_list[ref_index][precursor].shape[0])]
            # msms_dfs_dicts_list[ref_index][precursor]['id'] = id_list
        lookup_dicts[precursor]["Number of MS/MS"] = msms_dfs_dicts_list[ref_index][precursor].shape[0]
        id_list = [lookup_dicts[precursor]['Start id'] + i for i in range(lookup_dicts[precursor]["Number of MS/MS"])]
        msms_dfs_dicts_list[ref_index][precursor]['id'] = id_list
    

Processing TUM_first_pool_1_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_1_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_2_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_2_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_3_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_3_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_4_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_4_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_5_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_5_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_6_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_6_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_7_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_7_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_8_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_8_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_9_01_01_2xIT_2xHCD-1h-R1-tryptic


C:\Users\Admin\AppData\Local\Temp\ipykernel_27084\1689723878.py:23: DtypeWarning: Columns (0: Reverse) have mixed types. Specify dtype option on import or set low_memory=False.
  msms_df = pd.read_csv(


Processing TUM_first_pool_9_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_10_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_10_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_11_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_11_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_12_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_12_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_13_01_01_2xIT_2xHCD-1h-R1-tryptic


C:\Users\Admin\AppData\Local\Temp\ipykernel_27084\1689723878.py:23: DtypeWarning: Columns (0: Oxidation (M) Score Diffs, 1: Reverse) have mixed types. Specify dtype option on import or set low_memory=False.
  msms_df = pd.read_csv(


Processing TUM_first_pool_13_01_01_DDA-1h-R2-tryptic


...

Processing TUM_first_pool_16_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_17_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_17_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_18_01_01_2xIT_2xHCD-1h-R1-tryptic


C:\Users\Admin\AppData\Local\Temp\ipykernel_27084\1689723878.py:23: DtypeWarning: Columns (0: Reverse) have mixed types. Specify dtype option on import or set low_memory=False.
  msms_df = pd.read_csv(


Processing TUM_first_pool_18_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_19_01_01_2xIT_2xHCD-1h-R1-tryptic
Processing TUM_first_pool_19_01_01_DDA-1h-R2-tryptic
Processing TUM_first_pool_20_01_01_2xIT_2xHCD-1h-R1-tryptic


C:\Users\Admin\AppData\Local\Temp\ipykernel_27084\1689723878.py:23: DtypeWarning: Columns (0: Reverse) have mixed types. Specify dtype option on import or set low_memory=False.
  msms_df = pd.read_csv(


Processing TUM_first_pool_20_01_01_DDA-1h-R2-tryptic


In [29]:
def save_lookup_df(lookup_dicts, save_dir):
    try:
        os.makedirs(save_dir, exist_ok=True)
    except OSError as e:
        print(f"Error creating directory {save_dir}: {e}")

    for k, v in lookup_dicts.items():
        v |= {"Precursor" : k} 
    lookup_df = pd.DataFrame(lookup_dicts.values())
    lookup_df = lookup_df[["Precursor", "Reference table", "Start id", "Number of MS/MS"]]
    lookup_df.to_csv(os.path.join(save_dir, "lookup.tsv"), sep='\t', index=False)

In [30]:
def save_msms_dfs(msms_dfs_dicts_list, save_dir):
    try:
        os.makedirs(save_dir, exist_ok=True)
    except OSError as e:
        print(f"Error creating directory {save_dir}: {e}")
    for i in range(len(msms_dfs_dicts_list)):
        current_msms_df = pd.concat(msms_dfs_dicts_list[i].values(), ignore_index=True)
        current_msms_df.to_csv(os.path.join(save_dir, f"{i:04d}.tsv"), sep='\t', index=False)

In [31]:
save_lookup_df(lookup_dicts, "final_data")
save_msms_dfs(msms_dfs_dicts_list, "final_data")